# Stage 2 — Download the summary PDFs

For every device that filed a **Summary**, download its 510(k) summary PDF from FDA. These PDFs are
the only place predicate relationships are written down (the openFDA record has no predicate field).

**This is the long stage** (~1–3 hours for the full 154-code corpus). It checkpoints to a manifest,
so if it stops you can just re-run this notebook and it resumes where it left off.

Network: yes — but PDFs are static documents, so this is reproducible whenever run.

In [ ]:
import os, time, urllib.request, urllib.error
import pandas as pd

corpus = pd.read_csv("data/corpus.csv", dtype=str)
summ = corpus[corpus["statement_or_summary"] == "Summary"].copy()
os.makedirs("data/summaries", exist_ok=True)
MANIFEST = "data/download_manifest.csv"
print(f"{len(summ)} summary devices to fetch")

### Folder convention

FDA files summaries in fiscal-year folders encoded in the K-number's 2nd–3rd digits:
`K01…`–`K09…` → `pdf{N}` (no leading zero); `K10…`–`K25…` → `pdf{NN}`; pre-2001 / FY26+ → bare `pdf/`.

In [ ]:
def candidate_urls(k):
    yy = k[1:3]
    base = "https://www.accessdata.fda.gov/cdrh_docs"
    urls = []
    try:
        n = int(yy)
    except ValueError:
        n = None
    if n is not None and 1 <= n <= 9:
        urls.append(f"{base}/pdf{n}/{k}.pdf")
    if n is not None and 10 <= n <= 25:
        urls.append(f"{base}/pdf{yy}/{k}.pdf")
    urls.append(f"{base}/pdf/{k}.pdf")          # pre-2001 / fallback
    return urls

# resume support
done = set()
if os.path.exists(MANIFEST):
    done = set(pd.read_csv(MANIFEST, dtype=str)["k_number"])
rows = []
t0 = time.time()
for i, k in enumerate(summ["k_number"]):
    if k in done:
        continue
    status, path = 404, ""
    for url in candidate_urls(k):
        try:
            req = urllib.request.Request(url, method="GET", headers={"User-Agent": "Mozilla/5.0 (compatible; predicate-graph-study/1.0; +research)"})
            with urllib.request.urlopen(req, timeout=60) as r:
                data = r.read()
            if data[:4] == b"%PDF":
                path = f"data/summaries/{k}.pdf"
                open(path, "wb").write(data)
                status = 200
                break
        except urllib.error.HTTPError as e:
            status = e.code
        except Exception:
            status = -1
        time.sleep(0.05)
    rows.append({"k_number": k, "status": status, "path": path})
    if len(rows) % 200 == 0:
        pd.DataFrame(rows).to_csv(MANIFEST, mode="a", header=not os.path.exists(MANIFEST), index=False)
        rows = []
        print(f"  {i+1}/{len(summ)} processed, {time.time()-t0:.0f}s")
    time.sleep(0.2)
if rows:
    pd.DataFrame(rows).to_csv(MANIFEST, mode="a", header=not os.path.exists(MANIFEST), index=False)

man = pd.read_csv(MANIFEST, dtype=str)
ok = (man["status"] == "200").sum()
print(f"CHECKPOINT  manifest rows {len(man)} of {len(summ)}")
print(f"CHECKPOINT  retrieved {ok} ({100*ok/len(summ):.1f}%) — expect ~92%")